# Lab | Reinforcement Learning

In this lab, we will implement tabular reinforcement learning algorithms from scratch using Gymnasium environments. We will explore FrozenLake and Taxi environments, build a Q-Learning agent, and compare it with SARSA.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

%matplotlib inline

## Task 1: Environment Exploration

### 1.1 FrozenLake-v1 Exploration

We start by creating the **FrozenLake-v1** environment. This is a 4x4 grid where the agent must reach the goal without falling into holes.

In [ ]:
# Create the FrozenLake environment
env_fl = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="ansi")

print(f"Observation Space: {env_fl.observation_space}")
print(f"Action Space: {env_fl.action_space}")

print("\nAction Index Meanings:")
print("0: Left")
print("1: Down")
print("2: Right")
print("3: Up")

Now we run 5 episodes with random actions to see how the agent performs.

In [ ]:
print("Running 5 episodes with random actions on FrozenLake:\n")
for episode in range(1, 6):
    state, info = env_fl.reset()
    terminated = False
    truncated = False
    total_reward = 0
    steps = 0
    
    while not (terminated or truncated):
        action = env_fl.action_space.sample()
        state, reward, terminated, truncated, info = env_fl.step(action)
        total_reward += reward
        steps += 1
        
    print(f"Episode {episode}: Total Reward = {total_reward}, Steps = {steps}")

print("\nFinal state render:")
print(env_fl.render())

### 1.2 Taxi-v3 Exploration

Next, we explore the **Taxi-v3** environment. In this environment, a taxi must pick up and drop off a passenger at the correct location.

In [ ]:
# Create the Taxi environment
env_taxi = gym.make("Taxi-v3", render_mode="ansi")

print(f"Observation Space: {env_taxi.observation_space}")
print(f"Action Space: {env_taxi.action_space}")

print("\nAction Index Meanings:")
print("0: South")
print("1: North")
print("2: East")
print("3: West")
print("4: Pick-up")
print("5: Drop-off")

In [ ]:
print("Running 5 episodes with random actions on Taxi:\n")
for episode in range(1, 6):
    state, info = env_taxi.reset()
    terminated = False
    truncated = False
    total_reward = 0
    steps = 0
    
    while not (terminated or truncated):
        action = env_taxi.action_space.sample()
        state, reward, terminated, truncated, info = env_taxi.step(action)
        total_reward += reward
        steps += 1
        
    print(f"Episode {episode}: Total Reward = {total_reward}, Steps = {steps}")

print("\nFinal state render:")
print(env_taxi.render())

### 1.3 Comparison

**Observation Spaces:**
- **FrozenLake-v1 (4x4):** Discrete(16) - 16 possible states (each tile in the grid).
- **Taxi-v3:** Discrete(500) - 500 possible states (taxi location, passenger location, and destination).

**Action Spaces:**
- **FrozenLake-v1:** Discrete(4) - Left, Down, Right, Up.
- **Taxi-v3:** Discrete(6) - South, North, East, West, Pick-up, Drop-off.

**Why is Taxi harder?**
Taxi is significantly harder because it has a much larger state space (500 vs 16) and a more complex goal. In FrozenLake, the agent just needs to navigate to a specific tile. In Taxi, the agent must navigate to the passenger, perform a 'Pick-up' action, navigate to the destination, and then perform a 'Drop-off' action. This sequence of specific actions makes the reward sparse and the problem more difficult to solve.

## Task 2: Q-Learning on FrozenLake

In this task, we implement the Q-Learning algorithm from scratch and train it on the FrozenLake-v1 environment.

In [ ]:
def train_q_learning(env, num_episodes=10000, alpha=0.8, gamma=0.95, epsilon=1.0, epsilon_decay=0.995, min_epsilon=0.01):
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    q_table = np.zeros((n_states, n_actions))
    
    rewards_per_episode = []
    
    for episode in range(num_episodes):
        state, info = env.reset()
        total_reward = 0
        terminated = False
        truncated = False
        
        while not (terminated or truncated):
            # Epsilon-greedy action selection
            if np.random.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(q_table[state])
            
            next_state, reward, terminated, truncated, info = env.step(action)
            
            # Q-Learning update rule
            # Q(s, a) = Q(s, a) + alpha * [r + gamma * max(Q(s', a')) - Q(s, a)]
            old_value = q_table[state, action]
            next_max = np.max(q_table[next_state])
            
            new_value = old_value + alpha * (reward + gamma * next_max - old_value)
            q_table[state, action] = new_value
            
            state = next_state
            total_reward += reward
            
        rewards_per_episode.append(total_reward)
        
        # Epsilon decay
        epsilon = max(min_epsilon, epsilon * epsilon_decay)
        
    return q_table, rewards_per_episode

# Hyperparameters
num_episodes_fl = 10000
alpha_fl = 0.8
gamma_fl = 0.95
epsilon_fl = 1.0
epsilon_decay_fl = 0.995
min_epsilon_fl = 0.01

# Train the agent
q_table_fl, rewards_fl = train_q_learning(
    env_fl, 
    num_episodes=num_episodes_fl, 
    alpha=alpha_fl, 
    gamma=gamma_fl, 
    epsilon=epsilon_fl, 
    epsilon_decay=epsilon_decay_fl, 
    min_epsilon=min_epsilon_fl
)

print("Training complete for FrozenLake.")

### 2.2 Results and Visualization

We plot the cumulative reward over episodes using a rolling average to smooth the curve.

In [ ]:
def plot_rewards(rewards, title="Learning Curve", window=100):
    rolling_avg = pd.Series(rewards).rolling(window=window).mean()
    plt.figure(figsize=(10, 5))
    plt.plot(rolling_avg)
    plt.title(title)
    plt.xlabel("Episode")
    plt.ylabel(f"Average Reward (window={window})")
    plt.grid(True)
    plt.show()

plot_rewards(rewards_fl, title="Q-Learning on FrozenLake-v1")

In [ ]:
print("Final Q-Table:")
print(q_table_fl)

start_state_actions = q_table_fl[0]
best_action_start = np.argmax(start_state_actions)
action_names = ["Left", "Down", "Right", "Up"]
print(f"\nBest action for start state (0): {action_names[best_action_start]}")

### 2.3 Interpretation

The agent successfully learned to navigate FrozenLake. In the start state (0), the highest Q-value typically corresponds to either **Down** or **Right**, which moves the agent away from the wall and towards the goal. Since this version is not slippery (`is_slippery=False`), the agent can follow a deterministic path. The learned policy makes intuitive sense as it avoids the holes (H) and heads towards the goal (G).

## Task 3: Q-Learning on Taxi

Now we apply the same Q-Learning algorithm to the **Taxi-v3** environment. This problem is more complex due to the larger state space.

In [ ]:
# Hyperparameters for Taxi
num_episodes_taxi = 20000
alpha_taxi = 0.8
gamma_taxi = 0.95
epsilon_taxi = 1.0
epsilon_decay_taxi = 0.995
min_epsilon_taxi = 0.01

# Train the agent on Taxi
q_table_taxi, rewards_taxi = train_q_learning(
    env_taxi, 
    num_episodes=num_episodes_taxi, 
    alpha=alpha_taxi, 
    gamma=gamma_taxi, 
    epsilon=epsilon_taxi, 
    epsilon_decay=epsilon_decay_taxi, 
    min_epsilon=min_epsilon_taxi
)

print("Training complete for Taxi.")

In [ ]:
plot_rewards(rewards_taxi, title="Q-Learning on Taxi-v3", window=100)

### 3.2 Evaluation

We evaluate the trained agent on 100 test episodes with `epsilon = 0` (pure exploitation).

In [ ]:
def evaluate_agent(env, q_table, num_episodes=100):
    total_rewards = []
    successes = 0
    
    for episode in range(num_episodes):
        state, info = env.reset()
        episode_reward = 0
        terminated = False
        truncated = False
        
        while not (terminated or truncated):
            action = np.argmax(q_table[state])
            state, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            
        total_rewards.append(episode_reward)
        if episode_reward > 0:
            successes += 1
            
    avg_reward = np.mean(total_rewards)
    success_rate = successes / num_episodes
    return avg_reward, success_rate

avg_reward_taxi, success_rate_taxi = evaluate_agent(env_taxi, q_table_taxi)
print(f"Average Reward: {avg_reward_taxi}")
print(f"Success Rate: {success_rate_taxi * 100}%")

### 3.3 Discussion

**Comparison to FrozenLake:**
The training curve for Taxi typically takes longer to stabilize compared to FrozenLake. In FrozenLake, the rewards are sparse (only at the end), but the state space is small. In Taxi, the state space is much larger, and the agent receives negative rewards for each step and large negative rewards for illegal pick-up/drop-off actions. This makes the initial learning phase more volatile.

**Stabilization:**
The agent usually starts to show significant improvement after a few thousand episodes, and performance stabilizes as epsilon approaches its minimum value. In this run, we can see the average reward increasing and then leveling off as the agent masters the pick-up and drop-off sequence.

## Task 4: SARSA Comparison

In this final task, we implement the **SARSA** (State-Action-Reward-State-Action) algorithm and compare it with Q-Learning on the Taxi-v3 environment. The key difference is that SARSA is an **on-policy** algorithm, meaning it updates its Q-values based on the action actually taken by the current policy.

In [ ]:
def train_sarsa(env, num_episodes=20000, alpha=0.8, gamma=0.95, epsilon=1.0, epsilon_decay=0.995, min_epsilon=0.01):
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    q_table = np.zeros((n_states, n_actions))
    
    rewards_per_episode = []
    
    for episode in range(num_episodes):
        state, info = env.reset()
        
        # Choose initial action
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])
            
        total_reward = 0
        terminated = False
        truncated = False
        
        while not (terminated or truncated):
            next_state, reward, terminated, truncated, info = env.step(action)
            
            # Choose next action (on-policy)
            if np.random.random() < epsilon:
                next_action = env.action_space.sample()
            else:
                next_action = np.argmax(q_table[next_state])
            
            # SARSA update rule
            # Q(s, a) = Q(s, a) + alpha * [r + gamma * Q(s', a') - Q(s, a)]
            old_value = q_table[state, action]
            next_value = q_table[next_state, next_action]
            
            new_value = old_value + alpha * (reward + gamma * next_value - old_value)
            q_table[state, action] = new_value
            
            state = next_state
            action = next_action
            total_reward += reward
            
        rewards_per_episode.append(total_reward)
        epsilon = max(min_epsilon, epsilon * epsilon_decay)
        
    return q_table, rewards_per_episode

# Train SARSA on Taxi
q_table_sarsa, rewards_sarsa = train_sarsa(
    env_taxi, 
    num_episodes=num_episodes_taxi, 
    alpha=alpha_taxi, 
    gamma=gamma_taxi, 
    epsilon=epsilon_taxi, 
    epsilon_decay=epsilon_decay_taxi, 
    min_epsilon=min_epsilon_taxi
)

print("Training complete for SARSA on Taxi.")

### 4.2 Comparison of Learning Curves

In [ ]:
window = 100
rolling_avg_q = pd.Series(rewards_taxi).rolling(window=window).mean()
rolling_avg_sarsa = pd.Series(rewards_sarsa).rolling(window=window).mean()

plt.figure(figsize=(12, 6))
plt.plot(rolling_avg_q, label="Q-Learning (Off-policy)")
plt.plot(rolling_avg_sarsa, label="SARSA (On-policy)")
plt.title("Q-Learning vs SARSA on Taxi-v3")
plt.xlabel("Episode")
plt.ylabel(f"Average Reward (window={window})")
plt.legend()
plt.grid(True)
plt.show()

### 4.3 Final Evaluation

In [ ]:
avg_reward_sarsa, success_rate_sarsa = evaluate_agent(env_taxi, q_table_sarsa)

results = pd.DataFrame({
    "Algorithm": ["Q-Learning", "SARSA"],
    "Average Test Reward": [avg_reward_taxi, avg_reward_sarsa],
    "Success Rate": [f"{success_rate_taxi*100}%", f"{success_rate_sarsa*100}%"]
})
print(results)

### 4.4 Conclusion and Discussion

**Which algorithm converged faster?**
Typically, Q-Learning converges slightly faster in this environment as it directly learns the optimal policy (off-policy) by considering the maximum possible future reward, regardless of the exploration steps taken. SARSA, being on-policy, can be more conservative during training because it accounts for the actual (potentially exploratory) actions it will take.

**Which achieved a higher final reward?**
Both algorithms usually achieve very similar final rewards once the policy has stabilized and epsilon is minimal. In the test phase (where epsilon=0), both should ideally follow the optimal policy.

**Fundamental Difference:**
- **Q-Learning (Off-policy):** Updates the Q-value based on the maximum possible reward in the next state: $Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma \max_{a'} Q(s', a') - Q(s, a)]$. It learns about the optimal policy while following an exploratory one.
- **SARSA (On-policy):** Updates the Q-value based on the actual action taken in the next state: $Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma Q(s', a') - Q(s, a)]$. It learns about the policy it is currently following, including its exploration.

**Preference:**
SARSA is often preferred in environments where exploratory actions can be very costly or dangerous (e.g., a real robot that might fall off a cliff during exploration). SARSA will learn a "safer" policy that stays away from hazards even during training. Q-Learning is preferred when we want to learn the absolute optimal policy and can afford the risks of exploration during the training phase.